# GT |error| on AF2 models, with the matching NMR ensemble as grey reference

Aligns the AF2 models and ensembles with TM-align, then writes B-factor PDBs and one `.pml` per entry.

In [ ]:
import json
import os
import urllib.request

import numpy as np
import pandas as pd
from Bio.Align import PairwiseAligner
from Bio.PDB import PDBIO, PDBParser
from Bio.PDB.Polypeptide import three_to_index, index_to_one
from tmtools import tm_align

PRED_PATH = ("../../../data/analysis/Fig5/predictions_N_ensemble_10seed_sorted_centered_err.json")
GT_PATH = ("../../../data/rBMRB/cluster_30p/PANAV/test.json")
AF2_DIR = ("../../../data/rBMRB/cluster_30p/structures_split_filtered/test")
OUT_DIR = os.getcwd()
PDB_DIR = os.path.join(OUT_DIR, "pdb_raw")
os.makedirs(PDB_DIR, exist_ok=True)

# BMRB pid -> (PDB id from the same deposition/paper, PyMOL object prefix, note)
TARGETS = {
    "11547_1": ("2RU6", "ub_alt",  "UNDER-PRED, Q41N alternative state N2 (Kitahara)"),
    "30693_1": ("6V5D", "ub_eros", "WT, EROS3 RDC+NOE ensemble (Griesinger)"),
}
REF_AF2_PID = "30693_1"     # every AF2 model is aligned onto this one -> shared frame / camera

SENTINEL = -1.00
VMIN, VMAX = 0.0, 5.0       # must match draw_colorbar.py / the *_diff.pml renders
TOP_THRESH = 5.0            # GT |error| above this: AF2 side chain sticks + N sphere
MIN_IDENTITY = 0.95         # minimum identity over aligned positions, else refuse to write
N_MODELS = 10               # same number of NMR models (the first N) for every ensemble

In [ ]:
def gt_by_position(pid, pred_json):
    """{0-based BMRB sequence index -> ground-truth |error| (oof_resid)}, same mask & oof_mask &
    finite filtering used throughout the oof-error analysis."""
    mask = np.array(pred_json["mask"][pid], dtype=bool)
    oof_resid = np.array(pred_json["oof_resid"][pid], dtype=np.float64)
    oof_mask = np.array(pred_json["oof_mask"][pid], dtype=bool)
    n = min(len(mask), len(oof_resid), len(oof_mask))
    valid = mask[:n] & oof_mask[:n] & np.isfinite(oof_resid[:n])
    idx = np.where(valid)[0]
    return dict(zip(idx.tolist(), oof_resid[:n][idx].tolist()))


def fetch_pdb(pdb_id):
    path = os.path.join(PDB_DIR, f"{pdb_id}.pdb")
    if not os.path.exists(path):
        urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id}.pdb", path)
    return path


def one_letter(res_list):
    return "".join(index_to_one(three_to_index(r.get_resname())) for r in res_list)


def load_models(path, name, n_keep=None):
    """All models (capped at n_keep), first chain, standard amino acids only (drops waters/ligands/
    hetero residues). Returns (struct, per-model residue lists); every model must carry the same
    residues as model 1."""
    struct = PDBParser(QUIET=True).get_structure(name, path)
    if n_keep is not None:
        for m in list(struct)[n_keep:]:
            struct.detach_child(m.id)
    chain_id = next(struct[0].get_chains()).id
    model_res = []
    for model in struct:
        for c in list(model):
            if c.id != chain_id:
                model.detach_child(c.id)
        chain = model[chain_id]
        for res in list(chain):
            if res.id[0] != " ":
                chain.detach_child(res.id)
        model_res.append(list(chain))
    for k, res_k in enumerate(model_res[1:], start=2):
        if one_letter(res_k) != one_letter(model_res[0]):
            raise ValueError(f"{name}: model {k} residues differ from model 1")
    return struct, model_res


_aligner = PairwiseAligner()
_aligner.mode = "global"
_aligner.match_score, _aligner.mismatch_score = 2, -1
_aligner.open_gap_score, _aligner.extend_gap_score = -5, -0.5
_aligner.end_gap_score = 0   # free terminal overhangs (tags / truncated termini)


def map_to_bmrb(seq, bmrb_seq):
    """{residue list index -> BMRB 0-based index} + list of mismatches (i, j, aa, aa)."""
    aln = _aligner.align(seq, bmrb_seq)[0]
    mapping, mismatches = {}, []
    for (s1, e1), (s2, e2) in zip(*aln.aligned):
        for i, j in zip(range(s1, e1), range(s2, e2)):
            mapping[i] = j
            if seq[i] != bmrb_seq[j]:
                mismatches.append((i, j, seq[i], bmrb_seq[j]))
    return mapping, mismatches


def ca_array(res_list):
    """CA coords (N,3) + one-letter seq of the residues that have a CA, for TM-align."""
    res = [r for r in res_list if "CA" in r]
    return np.array([r["CA"].coord for r in res], dtype=np.float64), one_letter(res)


def apply_tm(tm, atoms):
    """Apply a TM-align result (mobile = chain 1) to atoms: x -> u @ x + t."""
    for atom in atoms:
        atom.set_coord(tm.u @ atom.coord.astype(np.float64) + tm.t)


def set_gt_bfactor(res_list, mapping, gt):
    """GT |error| on every atom of each mapped residue, SENTINEL where there is no data."""
    for i, res in enumerate(res_list):
        j = mapping.get(i)
        val = gt.get(j, SENTINEL) if j is not None else SENTINEL
        for atom in res:
            atom.set_bfactor(val)

## Sequence check: PDB ensemble and AF2 model vs BMRB sequence

In [ ]:
gt_json = json.load(open(GT_PATH))
inv = {v: k for k, v in gt_json["entry_ID"].items()}
pred_json = json.load(open(PRED_PATH))

entries = {}
for pid, (pdb_id, obj, note) in TARGETS.items():
    bmrb_seq = "".join(gt_json["sequence"][inv[pid]])
    pdb_struct, model_res = load_models(fetch_pdb(pdb_id), pdb_id, N_MODELS)
    if len(model_res) != N_MODELS:
        raise ValueError(f"{pdb_id} has only {len(model_res)} models < N_MODELS={N_MODELS}")
    af_struct, (af_res,) = load_models(os.path.join(AF2_DIR, f"{pid}.pdb"), f"{pid}_af2")
    print(f"{pid} <-> {pdb_id}, {len(model_res)} model(s) ({note})")
    maps = {}
    for label, res_list in [(pdb_id, model_res[0]), ("AF2", af_res)]:
        seq = one_letter(res_list)
        mapping, mismatches = map_to_bmrb(seq, bmrb_seq)
        identity = 1 - len(mismatches) / len(mapping)
        print(f"  {label:5s} len={len(seq)}  aligned={len(mapping)}/{len(bmrb_seq)}  identity={identity:.3f}")
        for i, j, a, b in mismatches:
            print(f"    MISMATCH {label} {res_list[i].get_resname()}{res_list[i].id[1]} ({a}) vs BMRB {j + 1} ({b})")
        if identity < MIN_IDENTITY:
            raise ValueError(f"{pid} vs {label}: identity {identity:.3f} < {MIN_IDENTITY}")
        maps[label] = mapping
    entries[pid] = dict(pdb_id=pdb_id, obj=obj, note=note, gt=gt_by_position(pid, pred_json),
                        pdb_struct=pdb_struct, model_res=model_res, pdb_map=maps[pdb_id],
                        af_struct=af_struct, af_res=af_res, af_map=maps["AF2"])

## Align and compute TM-score / RMSD

In [ ]:
# step 2: every AF2 model onto the reference AF2 model
ref_xyz, ref_seq = ca_array(entries[REF_AF2_PID]["af_res"])
for pid, e in entries.items():
    xyz, seq = ca_array(e["af_res"])
    tm = tm_align(xyz, ref_xyz, seq, ref_seq)
    apply_tm(tm, e["af_struct"].get_atoms())
    e["tm_af2_vs_ref"] = tm.tm_norm_chain2

rows = []
for pid, e in entries.items():
    # step 3: every ensemble model onto this entry's own AF2 model
    af_xyz, af_seq = ca_array(e["af_res"])
    af_ca = {e["af_map"][i]: r["CA"].coord.astype(np.float64)
             for i, r in enumerate(e["af_res"]) if i in e["af_map"] and "CA" in r}
    tms, rmsd_tm, rmsd_ca = [], [], []
    for res_k in e["model_res"]:
        xyz, seq = ca_array(res_k)
        tm = tm_align(xyz, af_xyz, seq, af_seq)          # mobile = PDB model, target = AF2
        apply_tm(tm, [a for r in res_k for a in r])
        tms.append(tm.tm_norm_chain1)                    # normalized by the experimental chain
        rmsd_tm.append(tm.rmsd)
        pdb_ca = {e["pdb_map"][i]: r["CA"].coord.astype(np.float64)
                  for i, r in enumerate(res_k) if i in e["pdb_map"] and "CA" in r}
        common = sorted(set(af_ca) & set(pdb_ca))
        d = np.array([af_ca[j] - pdb_ca[j] for j in common])
        rmsd_ca.append(float(np.sqrt((d ** 2).sum(1).mean())))

    # GT |error| into the B-factor column of both (AF2: replaces pLDDT), then write
    set_gt_bfactor(e["af_res"], e["af_map"], e["gt"])
    for res_k in e["model_res"]:
        set_gt_bfactor(res_k, e["pdb_map"], e["gt"])
    e["out_af2"] = f"{pid}_{e['pdb_id']}_AF2_gt_error_bfactor.pdb"
    e["out_pdb"] = f"{pid}_{e['pdb_id']}_gt_error_bfactor.pdb"
    for struct, out in [(e["af_struct"], e["out_af2"]), (e["pdb_struct"], e["out_pdb"])]:
        io = PDBIO()
        io.set_structure(struct)
        io.save(os.path.join(OUT_DIR, out))

    e.update(tm_mean=float(np.mean(tms)), n_models=len(tms))
    rows.append({"pdb_id": pid, "PDB": e["pdb_id"], "n_models": len(tms),
                 "tm_af2_vs_ref": e["tm_af2_vs_ref"],
                 "tm_mean": np.mean(tms), "tm_min": np.min(tms), "tm_max": np.max(tms),
                 "rmsd_tmalign_mean": np.mean(rmsd_tm),
                 "rmsd_ca_mean": np.mean(rmsd_ca), "rmsd_ca_min": np.min(rmsd_ca),
                 "rmsd_ca_max": np.max(rmsd_ca)})

tm_af2_df = pd.DataFrame(rows)
tm_af2_df.to_csv(os.path.join(OUT_DIR, "af2_vs_pdb_tmscore.csv"), index=False)
print(tm_af2_df.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

## PyMOL scripts

In [ ]:
PML_STYLE = """bg_color white
set ray_opaque_background, 0
set orthoscopic, on

set specular, 0
set reflect, 0
set ambient, 1
set depth_cue, 0
set ray_trace_mode, 1
set ray_trace_depth_factor, 1
set ray_trace_disco_factor, 1
set ray_trace_gain, 0.005
set ray_shadow, off
set antialias, 2
set cartoon_oval_length, 1
set cartoon_rect_length, 1
"""

for pid, e in entries.items():
    obj = e["obj"]
    af, ref_obj = f"{obj}_af2", f"{obj}_pdb"
    png = f"{pid}_{e['pdb_id']}_AF2_vs_PDB_gt_error.png"
    # NOTE: no ';' anywhere in the .pml (even in comments) -- PyMOL treats it as a command separator
    pml = f"""{PML_STYLE}
########################################
# {pid} -- AF2 model (colored) vs {e['pdb_id']} ensemble (grey), {e['note']}
# TM-score (norm. by {e['pdb_id']}) = {e['tm_mean']:.3f}, mean over {e['n_models']} models
# AF2 B-factor = ground-truth |error| (oof_resid, 15N, ppm), -1 = no data (grey70)
########################################
load {e['out_pdb']}, {ref_obj}_ens
load {e['out_af2']}, {af}
# one object per NMR model, grouped -> everything sits in a single frame (no multi-state movie)
split_states {ref_obj}_ens, prefix={ref_obj}_m
delete {ref_obj}_ens
group {ref_obj}, {ref_obj}_m*
dss {ref_obj}
dss {af}
hide everything
show cartoon, {ref_obj} or {af}

# experimental ensemble: light grey, semi-transparent (all models overlaid), drawn underneath
color grey60, {ref_obj}
set cartoon_transparency, 0.3, {ref_obj}

select {af}_nodata, {af} and b < -0.5
select {af}_hasdata, {af} and b > -0.5
color grey50, {af}_nodata
spectrum b, white_red, {af}_hasdata, minimum={VMIN:.1f}, maximum={VMAX:.1f}

select daf_top5, {af} and b > {TOP_THRESH:g}
show sticks, daf_top5 and (sidechain or name CA+N)
show spheres, daf_top5 and name N
hide everything, hydro
set stick_radius, 0.35, daf_top5
set sphere_scale, 0.35, daf_top5 and name N

orient {af}
deselect
#ray 1800, 1500
#png {png}, dpi=300
"""
    assert ";" not in pml
    pml_path = os.path.join(OUT_DIR, f"render_gt_error_{pid}_{e['pdb_id']}_vs_AF2.pml")
    with open(pml_path, "w") as fh:
        fh.write(pml)
    print("wrote", os.path.basename(pml_path))